# Notebook 00b — Descarga automática de imágenes Sentinel-2 desde Copernicus

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/miguepoloc/teledeteccion/blob/main/sesiones/sesion-02/colab/00_descarga_copernicus.ipynb)

## Maestría en Ingeniería — Universidad del Magdalena
### Sesión 2: Fallback si el docente no tiene el USB o la imagen falla

---

> **¿Cuándo usar este notebook?**  
> - Si el USB del docente falla o no está disponible
> - Si quieres descargar las imágenes antes de la clase para trabajar offline en SNAP
> - Si quieres explorar otras fechas o zonas diferentes a las de la clase
> 
> **Nota:** para el laboratorio principal en GEE/Colab, NO necesitas descargar nada.
> Los notebooks 01–04 cargan todo directamente desde GEE a través de internet.

**Dos métodos disponibles:**
1. **Método A (recomendado):** Copernicus Data Space API — interfaz moderna, las 4 imágenes del curso
2. **Método B (alternativo):** GEE Export — exportar directamente desde GEE a Google Drive

**Imágenes que necesitas:**
| Imagen | Período | Para qué |
|--------|---------|----------|
| L2A 2024 limpia | Enero–marzo 2024, <10% nubes | Laboratorio principal |
| L1C 2024 | Misma escena sin corrección atmosférica | Comparar TOA vs BOA |
| L2A 2024 nublada | Octubre–noviembre 2024 | Demostrar problema de nubes |
| L2A 2018 limpia | Enero–marzo 2018, <10% nubes | Análisis multitemporal |

## MÉTODO A — Copernicus Data Space API

Requiere una cuenta gratuita en https://dataspace.copernicus.eu

**Paso previo:** registrarse y tener usuario y contraseña.

In [ ]:
!pip install requests tqdm -q
import requests, os, json
from tqdm import tqdm
from pathlib import Path

# === CONFIGURACIÓN — CAMBIA ESTOS VALORES ===
USUARIO = 'tu_email@correo.com'    # Tu email de Copernicus Data Space
PASSWORD = 'tu_contraseña'          # Tu contraseña
CARPETA_DESCARGA = Path('./datos_sentinel2')  # Carpeta de destino
# ============================================

CARPETA_DESCARGA.mkdir(exist_ok=True)
print(f'Carpeta de descarga: {CARPETA_DESCARGA.absolute()}')

In [ ]:
def obtener_token(usuario, password):
    """Obtiene el token de acceso OAuth2 de Copernicus Data Space."""
    resp = requests.post(
        'https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token',
        data={
            'grant_type': 'password',
            'client_id': 'cdse-public',
            'username': usuario,
            'password': password
        }
    )
    if resp.status_code != 200:
        raise Exception(f'Error de autenticación: {resp.status_code} — verifica usuario y contraseña')
    return resp.json()['access_token']

token = obtener_token(USUARIO, PASSWORD)
print('Autenticación exitosa')

In [ ]:
def buscar_sentinel2(fecha_inicio, fecha_fin, nivel='S2MSI2A', max_nubes=15,
                     bbox='-74.2,10.5,-73.8,11.0', max_resultados=5):
    """
    Busca imágenes Sentinel-2 en el catálogo de Copernicus Data Space.
    
    nivel: 'S2MSI2A' para L2A (Surface Reflectance) o 'S2MSI1C' para L1C (TOA)
    bbox: longitud_oeste,latitud_sur,longitud_este,latitud_norte
    """
    params = {
        'Collection': f'SENTINEL-2',
        'ProductType': nivel,
        'StartDate': f'{fecha_inicio}T00:00:00.000Z',
        'CompletionDate': f'{fecha_fin}T23:59:59.999Z',
        'Footprint': f'geography(POLYGON(({bbox.split(",")[0]} {bbox.split(",")[1]},{bbox.split(",")[2]} {bbox.split(",")[1]},{bbox.split(",")[2]} {bbox.split(",")[3]},{bbox.split(",")[0]} {bbox.split(",")[3]},{bbox.split(",")[0]} {bbox.split(",")[1]})))',
        'cloudCover': f'[0,{max_nubes}]',
        'maxRecords': max_resultados,
        'orderBy': 'startDate',
        'sortOrder': 'desc'
    }
    url = 'https://catalogue.dataspace.copernicus.eu/resto/api/collections/Sentinel2/search.json'
    resp = requests.get(url, params=params)
    if resp.status_code != 200:
        raise Exception(f'Error en búsqueda: {resp.status_code}')
    return resp.json().get('features', [])


# Buscar las 4 imágenes del curso
print('Buscando imágenes del curso...')
print('='*60)

busquedas = [
    ('L2A 2024 limpia (laboratorio principal)',   '2024-01-01', '2024-03-31', 'S2MSI2A', 10),
    ('L1C 2024 (comparar TOA vs BOA)',            '2024-01-01', '2024-03-31', 'S2MSI1C', 10),
    ('L2A 2024 nublada (demo nubes)',             '2024-10-01', '2024-11-30', 'S2MSI2A', 80),
    ('L2A 2018 limpia (análisis multitemporal)',  '2018-01-01', '2018-03-31', 'S2MSI2A', 15),
]

catalogo = {}
for nombre, fi, ff, nivel, nubes in busquedas:
    resultados = buscar_sentinel2(fi, ff, nivel=nivel, max_nubes=nubes)
    if resultados:
        r = resultados[0]
        props = r['properties']
        print(f'\n✅ {nombre}')
        print(f'   ID: {r["id"]}')
        print(f'   Fecha: {props.get("startDate", "?")[:10]}')
        print(f'   Nubes: {props.get("cloudCover", "?"):.1f}%')
        print(f'   Tamaño: {props.get("resourceSize", 0)/1e9:.2f} GB')
        catalogo[nombre] = r
    else:
        print(f'\n⚠️  {nombre}: sin resultados con los filtros actuales')

print('\n' + '='*60)
print(f'Total encontradas: {len(catalogo)}/4')

In [ ]:
def descargar_imagen(producto_id, nombre_archivo, token, carpeta):
    """
    Descarga una imagen Sentinel-2 del Copernicus Data Space.
    Los archivos .zip contienen la carpeta .SAFE completa que necesita SNAP.
    """
    url = f'https://zipper.dataspace.copernicus.eu/odata/v1/Products({producto_id})/$value'
    headers = {'Authorization': f'Bearer {token}'}
    
    ruta = carpeta / f'{nombre_archivo}.zip'
    if ruta.exists():
        print(f'Ya existe: {ruta.name} — saltando descarga')
        return ruta
    
    print(f'Descargando {nombre_archivo}...')
    resp = requests.get(url, headers=headers, stream=True)
    total = int(resp.headers.get('content-length', 0))
    
    with open(ruta, 'wb') as f, tqdm(total=total, unit='B', unit_scale=True, desc=nombre_archivo) as bar:
        for chunk in resp.iter_content(chunk_size=8192):
            if chunk:
                f.write(chunk)
                bar.update(len(chunk))
    
    print(f'✅ Descargado: {ruta.name} ({ruta.stat().st_size/1e9:.2f} GB)')
    return ruta


# ¿Descargar ahora? Descomenta las líneas que necesites
# ADVERTENCIA: cada imagen pesa 600 MB – 1.2 GB

print('Para descargar, descomenta las líneas correspondientes en la celda de abajo')
print('y asegúrate de tener suficiente espacio en disco (mínimo 5 GB libres)')

In [ ]:
# DESCARGA — Descomenta las que necesites

# Renovar token primero (expira en 10 minutos)
token = obtener_token(USUARIO, PASSWORD)

nombres_descarga = {
    'L2A 2024 limpia (laboratorio principal)':  'S2_L2A_2024_limpia',
    'L1C 2024 (comparar TOA vs BOA)':           'S2_L1C_2024',
    'L2A 2024 nublada (demo nubes)':            'S2_L2A_2024_nublada',
    'L2A 2018 limpia (análisis multitemporal)': 'S2_L2A_2018_limpia',
}

# Descomenta la(s) que quieras descargar:
# descargar_imagen(catalogo['L2A 2024 limpia (laboratorio principal)']['id'], nombres_descarga['L2A 2024 limpia (laboratorio principal)'], token, CARPETA_DESCARGA)
# descargar_imagen(catalogo['L1C 2024 (comparar TOA vs BOA)']['id'], nombres_descarga['L1C 2024 (comparar TOA vs BOA)'], token, CARPETA_DESCARGA)
# descargar_imagen(catalogo['L2A 2024 nublada (demo nubes)']['id'], nombres_descarga['L2A 2024 nublada (demo nubes)'], token, CARPETA_DESCARGA)
# descargar_imagen(catalogo['L2A 2018 limpia (análisis multitemporal)']['id'], nombres_descarga['L2A 2018 limpia (análisis multitemporal)'], token, CARPETA_DESCARGA)

print('Líneas comentadas. Para descargar, quita el # de la línea correspondiente.')

---

## MÉTODO B — Exportar desde GEE a Google Drive

Si no quieres crear cuenta en Copernicus, puedes exportar directamente desde GEE.
La imagen llega a tu Google Drive como un GeoTIFF (no un .SAFE completo — sin metadatos de SNAP,
pero funcional para QGIS y análisis Python).

In [ ]:
import ee
import geemap

try:
    ee.Initialize(project='teledeteccion-miguepoloc')
except Exception:
    ee.Authenticate()
    ee.Initialize(project='teledeteccion-miguepoloc')

zona_cacaotera = ee.Geometry.Rectangle([-74.2, 10.5, -73.8, 11.0])

def crear_composicion(fecha_inicio, fecha_fin, max_nubes=15):
    return (
        ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
        .filterBounds(zona_cacaotera)
        .filterDate(fecha_inicio, fecha_fin)
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', max_nubes))
        .median()
        .select(['B2','B3','B4','B5','B8','B8A','B11','B12'])
        .clip(zona_cacaotera)
    )

imagenes_exportar = {
    'S2_L2A_2024_limpia': crear_composicion('2024-01-01', '2024-03-31', 10),
    'S2_L2A_2024_nublada': crear_composicion('2024-10-01', '2024-11-30', 80),
    'S2_L2A_2018_limpia': crear_composicion('2018-01-01', '2018-03-31', 15),
}

print('Exportando imágenes a Google Drive — carpeta: teledeteccion_maestria')
tareas = []
for nombre, img in imagenes_exportar.items():
    task = ee.batch.Export.image.toDrive(
        image=img,
        description=nombre,
        folder='teledeteccion_maestria',
        fileNamePrefix=nombre,
        region=zona_cacaotera,
        scale=10,
        crs='EPSG:4326',
        maxPixels=1e9
    )
    task.start()
    tareas.append((nombre, task))
    print(f'  ✅ Tarea iniciada: {nombre} — Estado: {task.status()["state"]}')

print('\nLas exportaciones corren en segundo plano en los servidores de GEE.')
print('Ve a https://code.earthengine.google.com/tasks para ver el progreso.')
print('Cuando terminen (5-30 min), estarán en tu Google Drive.')

---

## Notas importantes

**¿Por qué 2018 y no 2015?**  
Sentinel-2A se lanzó el 23 de junio de 2015. No existen imágenes Sentinel-2 de enero-marzo 2015.
El primer año completo con buena cobertura de la zona es 2016-2017. Usamos **2018** como año
base por tener la mayor densidad de imágenes sin nubes en temporada seca en la SNSM.

**¿Cuánto espacio necesito?**  
- Archivo .SAFE completo (Copernicus): ~700 MB – 1.2 GB por imagen
- GeoTIFF de la zona cacaotera (GEE Export): ~50–150 MB por imagen (mucho más manejable)

**¿El GeoTIFF de GEE funciona en SNAP?**  
Sí, SNAP puede abrir GeoTIFFs. La diferencia con el .SAFE es que no tendrás los metadatos
del sensor ni la banda SCL (máscara de nubes). Para el laboratorio de índices funciona igual.